# 04 - EDA with SQL
**IBM Applied Data Science Capstone - SpaceX Falcon 9 First-Stage Landing Prediction**

Author: Pranav Vyankatesh Jagdale

**Goal:** load the launch table into SQLite and answer the lab questions with SQL.

> Run every cell in order. Keep your own outputs - they feed your final slides.

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.columns = [c.strip() for c in df.columns]
print(df.columns.tolist())
df.head()

['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


Column names differ slightly between lab versions. Check the printout above, then edit the rename dictionary if needed.

In [2]:
rename = {'Time (UTC)': 'Time'}
df = df.rename(columns=rename)
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')
con = sqlite3.connect('my_spacex.db')
df.to_sql('SPACEXTBL', con, if_exists='replace', index=False)

def q(sql):
    return pd.read_sql_query(sql, con)

### Task 1 - Unique launch sites

In [3]:
q('SELECT DISTINCT Launch_Site FROM SPACEXTBL')

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### Task 2 - Five records where the launch site starts with 'CCA'

In [4]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

,Date,Time,Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Task 3 - Total payload mass carried for NASA (CRS)

In [5]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL WHERE Customer LIKE 'NASA (CRS)'")

,total_payload_kg
0,45596


### Task 4 - Average payload mass for booster version F9 v1.1

In [6]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%'")

,avg_payload_kg
0,2534.666667


### Task 5 - Date of the first successful landing on a ground pad

In [7]:
q("SELECT MIN(Date) AS first_ground_pad_success FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (ground pad)'")

,first_ground_pad_success
0,2015-12-22


### Task 6 - Boosters with a successful drone-ship landing and payload between 4000 and 6000 kg

In [8]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome LIKE 'Success (drone ship)' AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### Task 7 - Total number of each mission outcome

In [9]:
q('SELECT Mission_Outcome, COUNT(*) AS n FROM SPACEXTBL GROUP BY Mission_Outcome')

,Mission_Outcome,n
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


### Task 8 - Boosters that carried the maximum payload

In [10]:
q('SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)')

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


### Task 9 - 2015 drone-ship failures: month, landing outcome, booster, launch site

In [11]:
q("""SELECT substr(Date, 6, 2) AS month, Landing_Outcome, Booster_Version, Launch_Site
     FROM SPACEXTBL
     WHERE Landing_Outcome LIKE 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015'""")

,month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Task 10 - Rank landing outcomes between 2010-06-04 and 2017-03-20, descending

In [12]:
q("""SELECT Landing_Outcome, COUNT(*) AS n
     FROM SPACEXTBL
     WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
     GROUP BY Landing_Outcome
     ORDER BY n DESC""")

,Landing_Outcome,n
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1


## Write your own interpretation here
One or two sentences per task: what does each result tell you about SpaceX launches? Note anything in the text columns that needed cleaning.